# 🗂️ Notebook 3: Checkpoints and Log Compaction

In notebook 1 we built a WAL that appends every write forever. That solves durability, but it creates a new problem:

> If the log grows forever, restart time grows forever too. After a year of writes, do we really want to replay a billion records every time the process starts?

Real systems fix this with two tricks that work together:

1. 📸 **Checkpoint** — every so often, save a snapshot of the current state.
2. ✂️ **Log truncation / compaction** — once the snapshot is safely on disk, throw away the old log records it covers.

On restart: load the latest snapshot, then replay only the tail of the log (the records written *after* the checkpoint). That's how Postgres, SQLite, RocksDB, Kafka and many others keep recovery fast even after months of writes.

## Learning objectives
- See why an unbounded WAL is a problem.
- Add a periodic checkpoint that snapshots state to disk.
- Truncate the old log safely and recover from `snapshot + tail-of-log`.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/write-ahead-log
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). Reload the window if it doesn't appear: `Cmd+Shift+P` → **Reload Window**.

In [ ]:
import os, json, tempfile, shutil, time

WORKDIR = tempfile.mkdtemp(prefix="wal_ckpt_")
print("workdir:", WORKDIR)

## 🟥 The problem: an ever-growing log

Let's append 50,000 records and time how long recovery takes. This is our baseline.

In [ ]:
big_log = os.path.join(WORKDIR, "big.log")

with open(big_log, "a") as f:
    for i in range(50_000):
        f.write(json.dumps({"op": "put", "k": f"k{i % 1000}", "v": i}) + "\n")
    f.flush(); os.fsync(f.fileno())

print("log size:", os.path.getsize(big_log), "bytes")

def replay(path):
    state = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            rec = json.loads(line)
            if rec["op"] == "put": state[rec["k"]] = rec["v"]
            elif rec["op"] == "del": state.pop(rec["k"], None)
    return state

t0 = time.perf_counter()
state = replay(big_log)
t1 = time.perf_counter()

print(f"replayed {os.path.getsize(big_log)//1024} KB in {(t1-t0)*1000:.1f} ms, {len(state)} live keys")
print("Notice: we only have 1000 unique keys, but we replayed 50,000 records to find that out.")

That's the core waste: the **live state** is tiny (1000 keys) but the **history** is huge. Most records are overwritten before we even read them.

## 🟩 The fix: checkpoint + truncate

Here's the safe recipe. Order matters — getting it wrong can lose data.

1. Write the current state to `state.snap.tmp` and `fsync` it.
2. Atomically rename `state.snap.tmp` → `state.snap` (rename is atomic on POSIX).
3. **Only now** truncate the old WAL file. Any records that arrived *before* the snapshot were captured; any records that arrived *after* the snapshot are still in the WAL.

If we crash between steps 2 and 3 we just have a snapshot plus a slightly-too-long log — still correct, we just replay a bit more than needed. If we crash between 1 and 2 we still have the old snapshot and the full log — still correct.

In [ ]:
class WalKVCheckpointed:
    def __init__(self, workdir):
        self.workdir = workdir
        self.log_path  = os.path.join(workdir, "wal.log")
        self.snap_path = os.path.join(workdir, "state.snap")
        self.data = {}
        self._recover()
        self._log = open(self.log_path, "a")

    # ------------- recovery -------------
    def _recover(self):
        # 1) start from the latest snapshot, if any
        if os.path.exists(self.snap_path):
            with open(self.snap_path) as f:
                self.data = json.load(f)
            print(f"  loaded snapshot with {len(self.data)} keys")
        # 2) then replay whatever is in the log (records after the snapshot)
        if os.path.exists(self.log_path):
            replayed = 0
            with open(self.log_path) as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    try:
                        rec = json.loads(line)
                    except json.JSONDecodeError:
                        print("  ⚠️ ignoring torn tail line"); continue
                    if rec["op"] == "put": self.data[rec["k"]] = rec["v"]
                    elif rec["op"] == "del": self.data.pop(rec["k"], None)
                    replayed += 1
            print(f"  replayed {replayed} log records")

    # ------------- writes -------------
    def _append(self, rec):
        self._log.write(json.dumps(rec) + "\n")
        self._log.flush(); os.fsync(self._log.fileno())

    def put(self, k, v):
        self._append({"op": "put", "k": k, "v": v})
        self.data[k] = v

    def delete(self, k):
        self._append({"op": "del", "k": k})
        self.data.pop(k, None)

    # ------------- checkpoint -------------
    def checkpoint(self):
        """Snapshot current state, then truncate the WAL. Safe order matters."""
        tmp = self.snap_path + ".tmp"
        with open(tmp, "w") as f:
            json.dump(self.data, f)
            f.flush(); os.fsync(f.fileno())
        os.replace(tmp, self.snap_path)           # atomic rename on POSIX
        # Flush directory metadata so the rename is durable too:
        dfd = os.open(self.workdir, os.O_RDONLY)
        try: os.fsync(dfd)
        finally: os.close(dfd)
        # Only now is it safe to drop the log:
        self._log.close()
        open(self.log_path, "w").close()          # truncate to zero
        self._log = open(self.log_path, "a")
        print(f"  checkpoint: {len(self.data)} keys snapshotted, WAL truncated")

    def close(self): self._log.close()

## 🏎️ Demo: write, checkpoint, write more, crash, recover

Write some records, take a checkpoint, write more records, then simulate a crash by just dropping the object. On "restart" (new instance), we should see the snapshot + the log tail combine into the correct state.

In [ ]:
store_dir = os.path.join(WORKDIR, "store")
os.makedirs(store_dir, exist_ok=True)

print("--- first boot ---")
kv = WalKVCheckpointed(store_dir)

for i in range(5):
    kv.put(f"k{i}", i)
print("  wrote k0..k4. log size:", os.path.getsize(kv.log_path))

kv.checkpoint()
print("  after checkpoint, log size:", os.path.getsize(kv.log_path), "snap size:", os.path.getsize(kv.snap_path))

# Writes that happen AFTER the checkpoint live only in the WAL until the next checkpoint.
kv.put("k5", 500)
kv.delete("k0")
print("  wrote k5 and deleted k0. log size:", os.path.getsize(kv.log_path))

# Simulate a crash: just abandon the object without closing (no extra flush).
del kv

print("\n--- restart ---")
kv2 = WalKVCheckpointed(store_dir)
print("recovered state:", kv2.data)
assert kv2.data == {"k1": 1, "k2": 2, "k3": 3, "k4": 4, "k5": 500}
print("✅ correct: snapshot + log tail reproduced exact state")
kv2.close()

## 📈 Why this matters in practice

Without checkpoints, startup time is O(all writes ever). With periodic checkpoints, startup time is bounded by **(size of live state) + (writes since last checkpoint)**.

Most databases run checkpoints:
- on a timer (e.g. every few minutes),
- or when the log grows past a size threshold,
- or both.

The trade-off is: more frequent checkpoints → faster recovery but more I/O during normal operation. Less frequent → cheaper steady-state but slower startup and more disk used by the log.

In [ ]:
shutil.rmtree(WORKDIR); print("cleaned up")

## ✅ Recap

- An unbounded WAL means unbounded recovery time.
- A **checkpoint** snapshots the current in-memory state to disk, and then the pre-checkpoint log can be discarded.
- The ordering `fsync snapshot → atomic rename → fsync dir → truncate log` is what keeps crash-safety intact throughout.
- This is exactly the pattern used by Postgres (`pg_wal` + checkpoints), SQLite (WAL mode + checkpoint), RocksDB (memtable → SST flush + WAL truncation), and Kafka (log compaction for keyed topics).